# F0 · Three-seed equal-probability ensemble
Self-contained Colab notebook: same official frozen DINOv2-S/14, P3 augmentation seed123, and E4 recipe as F0. Train fresh heads with seeds42/123/2026 against one fixed feature cache, then average all three probabilities equally. No TTA, fitted ensemble weights, seed subset search, external training data, or automatic Kaggle submission. Validation metrics are reported; improvement is not required to export this predeclared candidate. Historical Kaggle0.95510 is a reference, not a promised result.

## 0. Environment
Choose Runtime → Change runtime type → T4 GPU. Keep Colab’s installed compatible torch/torchvision pair.

In [ ]:
%pip -q install scikit-learn pandas pillow tqdm matplotlib kagglehub


In [ ]:
import os, platform, torchvision
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from datetime import datetime, timezone
import uuid
import random, hashlib, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
from IPython.display import display



## 1. Data and shared cache
Choose T4 GPU. Enable Colab Secret KAGGLE_API_TOKEN or disable downloading and set DATA_ROOT. This notebook trains three heads; MODE stays train_f0. Optional Drive preserves features/results. It reuses the compatible cache under student_outputs_F0_TTA/cache from the F0 notebook. Outputs go to a new student_outputs_F0_Ensemble run directory.

In [ ]:
MODE = 'train_f0'  # Required: shared feature extraction + fresh seeded heads.
HEAD_SEEDS = [42, 123, 2026]
assert HEAD_SEEDS == [42, 123, 2026]
USE_DRIVE = False
DOWNLOAD_FROM_KAGGLE = True
DATA_ROOT = Path('/content/car_data')
STORAGE_ROOT = Path('/content')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE_ROOT = Path('/content/drive/MyDrive/Car-Classification-Benchmark')
if DOWNLOAD_FROM_KAGGLE:
    import kagglehub
    if not os.environ.get('KAGGLE_API_TOKEN'):
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
        except Exception as exc:
            raise RuntimeError('Enable Colab Secret KAGGLE_API_TOKEN, or disable download and set DATA_ROOT to the extracted course data.') from exc
    DATA_ROOT = Path(kagglehub.competition_download('idnn-swe-3052-assignment-3-flowers-classification'))
assert MODE == 'train_f0'
for split in ('train','val','test'):
    if not (DATA_ROOT/split).is_dir(): raise FileNotFoundError(f'Missing {DATA_ROOT/split}; set DATA_ROOT or enable the download.')
assert (DATA_ROOT/'sample_submission.csv').is_file()
SEED_AUG,SEED_HEAD = 123,2026
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 16
EXTRACTION_BATCH_SIZE = BATCH_SIZE
EXPERIMENT = 'P3_geometry_multiview'
EPOCHS,LEARNING_RATE,WEIGHT_DECAY,PATIENCE = 35,1e-3,1e-3,7
MODEL_NAME = 'dinov2_vits14'
FEATURE_DIM = 384
HUB_REF = '7764ea0f912e53c92e82eb78a2a1631e92725fc8'
HUB_REPO = f'facebookresearch/dinov2:{HUB_REF}'
WEIGHTS_SOURCE = 'https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth'
N_AUG,INCLUDE_ORIGINAL = 4,True
PIPELINE_VERSION = 'F0-TTA-v1'
FORCE_REEXTRACT = False
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]
OUTPUT_DIR = STORAGE_ROOT/'student_outputs_F0_Ensemble'/RUN_ID
CACHE_DIR = STORAGE_ROOT/'student_outputs_F0_TTA'/'cache'
OUTPUT_DIR.mkdir(parents=True,exist_ok=False);CACHE_DIR.mkdir(parents=True,exist_ok=True)
def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)
set_seed(SEED_AUG)
print('MODE:',MODE,'Data:',DATA_ROOT,'Output:',OUTPUT_DIR,'Device:',DEVICE)
print('Python:',platform.python_version(),'PyTorch:',torch.__version__)

## 2. Dataset and P3 preprocessing
20prefixed classes,700training/200validation images. P3 uses one deterministic view plus four random geometric views; no ColorJitter. Validation/test always use bicubic Resize256→CenterCrop224 and ImageNet normalization.

In [ ]:
# preprocessing change
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1),
        interpolation=InterpolationMode.BICUBIC
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

val_transform = transforms.Compose([
    transforms.Resize(
        256,
        interpolation=InterpolationMode.BICUBIC
    ),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])


train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=train_transform)
val_ds = datasets.ImageFolder(DATA_ROOT / 'val', transform=val_transform)
preprocess = val_transform

assert train_ds.class_to_idx == val_ds.class_to_idx, 'train/ and val/ class folders must match'
NUM_CLASSES = len(train_ds.classes)
print('Classes:', NUM_CLASSES, '| Train:', len(train_ds), '| Val:', len(val_ds))
print('First class folders:', train_ds.classes[:5])
assert NUM_CLASSES == 20, f'Expected 20 classes, got {NUM_CLASSES}'

# Class-folder names from the instructor's export are prefixed by 000_, 001_, ...
# These prefixes make ImageFolder indices equal the Kaggle integer labels.
for i, folder in enumerate(train_ds.classes):
    assert folder.startswith(f'{i:03d}_'), f'Class-label mismatch at {folder}; ask your instructor'

assert (len(train_ds), len(val_ds)) == (700, 200), 'Expected original 700/200 split.'

## 3. Original sample-submission ordering
Test labels are never used. The original CSV is required; string IDs, uniqueness and file coverage are checked.

In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
test_dir = DATA_ROOT / 'test'
test_files = sorted(p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS)
assert test_files, f'No images found inside {test_dir.resolve()}'

sample_path = DATA_ROOT / 'sample_submission.csv'
assert sample_path.exists(), 'Original sample_submission.csv is required.'
if sample_path.exists():
    sample = pd.read_csv(sample_path, dtype={'image_id': str})
    assert {'image_id','label'}.issubset(sample.columns)
    required_ids = sample['image_id'].astype(str).tolist()
else:
    required_ids = sorted(p.stem for p in test_files)

by_id = {p.stem: p for p in test_files}
assert len(by_id) == len(test_files), 'Two test filenames share the same image_id stem'
assert len(set(required_ids)) == len(required_ids), 'Duplicate image_id in sample_submission.csv'
assert set(required_ids) == set(by_id), ('Test image IDs do not match sample_submission.csv',
                                         len(set(required_ids)-set(by_id)), len(set(by_id)-set(required_ids)))
print('Unlabeled test images:', len(required_ids))
print('Submission ID example:', required_ids[:3])

class TestImages(Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, index):
        p = by_id[self.ids[index]]
        with Image.open(p) as im: image = preprocess(im.convert('RGB'))
        return image

test_ds = TestImages(required_ids)
assert len(required_ids)==len(sample)


## 4. Official pretrained frozen DINOv2-S/14
Use the same permitted SSL architecture as F0,384-D CLS features, pinned official upstream source, no random-weight fallback. Hub caches downloads. No fine-tuning.

In [ ]:
backbone = torch.hub.load(HUB_REPO,MODEL_NAME,pretrained=True,trust_repo=True,skip_validation=True).to(DEVICE)
backbone.requires_grad_(False).eval()
assert backbone.embed_dim==FEATURE_DIM
with torch.inference_mode(): probe=backbone(torch.zeros(1,3,224,224,device=DEVICE))
assert probe.shape==(1,384)
del probe
def weight_digest(model):
    h=hashlib.sha256()
    for name,value in sorted(model.state_dict().items()):
        h.update(name.encode());h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
WEIGHTS_SHA256 = weight_digest(backbone)
print('Frozen backbone:',MODEL_NAME,'Feature dim:',FEATURE_DIM)


## 5. Safe P3 feature cache · training mode only
Cache identity includes actual pretrained weights/source, transforms, augmentation seed, view counts/layout, class mapping and content-hashed manifests. Invalid caches regenerate. Head-only changes reuse features. Source/metadata checks prevent stale in-memory features. No validation/test statistics enter fitting.

In [ ]:
if MODE == 'train_f0':
    def file_sha256(path):
        h = hashlib.sha256()
        with open(path, 'rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''): h.update(chunk)
        return h.hexdigest()

    def feature_metadata():
        ordered = ([Path(p) for p, _ in train_ds.samples],
                   [Path(p) for p, _ in val_ds.samples], [by_id[i] for i in required_ids])
        manifests = [[{'path': str(p.relative_to(DATA_ROOT)), 'sha256': file_sha256(p)}
                      for p in paths] for paths in ordered]
        sets = [{entry['sha256'] for entry in split} for split in manifests]
        assert not any(sets[i] & sets[j] for i,j in ((0,1),(0,2),(1,2))), 'Duplicate image content across splits'
        return {'pipeline': PIPELINE_VERSION, 'backbone': MODEL_NAME, 'feature_dim': FEATURE_DIM,
                'hub_repo': HUB_REPO, 'weights_source': WEIGHTS_SOURCE, 'weights_sha256': WEIGHTS_SHA256,
                'preprocessing': 'P3_geometry_multiview', 'train_transform': repr(train_transform),
                'val_transform': repr(val_transform), 'seed_aug': SEED_AUG, 'n_aug': N_AUG,
                'include_original': INCLUDE_ORIGINAL, 'layout': 'view-major',
                'class_to_idx': dict(train_ds.class_to_idx), 'manifest': manifests,
                'test_ids': list(required_ids), 'batch_size': EXTRACTION_BATCH_SIZE,
                'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
                'pillow': Image.__version__, 'device': DEVICE}

    def metadata_digest(meta):
        return hashlib.sha256(json.dumps(meta, sort_keys=True).encode()).hexdigest()

    @torch.inference_mode()
    def extract(loader, desc):
        backbone.eval()
        assert not any(p.requires_grad for p in backbone.parameters())
        embeddings = []
        for batch in tqdm(loader, desc=desc):
            x = batch[0] if isinstance(batch, (tuple, list)) else batch
            z = backbone(x.to(DEVICE)).float().cpu()
            assert z.ndim == 2 and z.shape[1] == FEATURE_DIM and torch.isfinite(z).all()
            embeddings.append(z)
        return torch.cat(embeddings)

    def valid_cache(payload, meta):
        try:
            assert payload['metadata'] == meta
            views = N_AUG + int(INCLUDE_ORIGINAL)
            for key,n in [('X_train',len(train_ds)*views), ('X_val',len(val_ds)), ('X_test',len(required_ids))]:
                x = payload[key]
                assert x.dtype == torch.float32 and x.shape == (n,FEATURE_DIM) and torch.isfinite(x).all()
            assert torch.equal(payload['y_train'], torch.tensor(train_ds.targets).repeat(views))
            assert torch.equal(payload['y_val'], torch.tensor(val_ds.targets))
            return True
        except (KeyError, AssertionError, TypeError, AttributeError):
            return False

    def get_features(force=False):
        meta = feature_metadata()
        signature = metadata_digest(meta)
        path = CACHE_DIR / f'F0_dinov2_vits14_p3_aug{SEED_AUG}_{signature}.pt'
        if path.exists() and not force:
            try:
                payload = torch.load(path, map_location='cpu', weights_only=True)
                if valid_cache(payload, meta):
                    print('Loaded compatible cache:', path)
                    return payload, signature, path
                print('Incompatible cache; regenerating')
            except Exception as exc:
                print('Unreadable cache; regenerating:', type(exc).__name__)
        set_seed(SEED_AUG)
        kwargs = dict(batch_size=EXTRACTION_BATCH_SIZE, shuffle=False, num_workers=0,
                      pin_memory=(DEVICE == 'cuda'))
        views = []
        original_transform = train_ds.transform
        try:
            if INCLUDE_ORIGINAL:
                train_ds.transform = val_transform
                views.append(extract(DataLoader(train_ds, **kwargs), 'Original training view'))
            train_ds.transform = train_transform
            for index in range(N_AUG):
                views.append(extract(DataLoader(train_ds, **kwargs), f'Augmented training view {index+1}'))
        finally:
            train_ds.transform = original_transform
        payload = {'metadata': meta, 'X_train': torch.cat(views),
                   'X_val': extract(DataLoader(val_ds, **kwargs), 'Validation'),
                   'X_test': extract(DataLoader(test_ds, **kwargs), 'Test'),
                   'y_train': torch.tensor(train_ds.targets).repeat(N_AUG + int(INCLUDE_ORIGINAL)),
                   'y_val': torch.tensor(val_ds.targets)}
        assert valid_cache(payload, meta)
        temp = path.with_suffix('.' + uuid.uuid4().hex + '.tmp')
        torch.save(payload, temp); temp.replace(path)
        print('Freshly extracted:', path)
        return payload, signature, path

    features, FEATURES_SIGNATURE, CACHE_PATH = get_features(FORCE_REEXTRACT)
    X_train, X_val, X_test = [features[k] for k in ('X_train','X_val','X_test')]
    y_train, y_val = features['y_train'], features['y_val']
    assert X_train.shape == (3500,384) and y_train.shape == (3500,)
    assert X_val.shape == (200,384) and X_test.shape == (len(sample),384)
    print(X_train.shape, X_val.shape, X_test.shape)
    FEATURES_MODEL_NAME = MODEL_NAME
    dataset_signature = lambda: metadata_digest(feature_metadata())
else:
    print('Checkpoint mode: feature extraction and classifier training will be skipped.')

## 6. Train-only standardization and E4 head
Head: Linear384→128, BatchNorm, ReLU, Dropout.3, Linear128→20. Default training uses AdamW lr/decay.001, batch64,35epochs, patience7; accuracy-only checkpoint selection matches F0. Fresh head/optimizer with head seed2026.

In [ ]:
HEAD_CONFIGS={'E4_mlp_bn_drop03':{'hidden_dim':128,'use_bn':True,'dropout':0.3}}
def build_head(experiment,input_dim,num_classes):
    assert experiment=='E4_mlp_bn_drop03' and input_dim==384 and num_classes==20
    return nn.Sequential(nn.Linear(input_dim,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,num_classes))
if MODE=='train_f0':
    feature_mean = X_train.mean(dim=0)
    feature_std = X_train.std(dim=0, unbiased=False).clamp_min(1e-6)
    Z_train = (X_train - feature_mean) / feature_std
    Z_val = (X_val - feature_mean) / feature_std
    Z_test = (X_test - feature_mean) / feature_std
    print('Features standardized without validation/test leakage.')

## 7. Three fresh classifiers
Backbone/features/statistics are shared; every head and optimizer are initialized afresh immediately after resetting its classifier seed. Shuffling has its own seeded generator. Each head restores its first strictly best validation-accuracy epoch. No seed is discarded.

In [ ]:
def train_one_head(seed):
    SEED_HEAD = seed
    from torch.utils.data import TensorDataset
    training_loader = DataLoader(TensorDataset(Z_train, y_train), batch_size=64,
                                 shuffle=True, num_workers=0)
    HEAD_EXPERIMENT = "E4_mlp_bn_drop03"
    
    # Ensure features match the selected backbone
    assert FEATURES_MODEL_NAME == MODEL_NAME
    assert FEATURES_SIGNATURE == dataset_signature(), (
        "Feature configuration changed. "
        "Rerun Sections 5 and 6."
    )
    
    # Reset classifier randomness
    random.seed(SEED_HEAD)
    np.random.seed(SEED_HEAD)
    torch.manual_seed(SEED_HEAD)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED_HEAD)
    
    # Reproducible training data order
    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED_HEAD)
    
    training_loader = DataLoader(
        TensorDataset(Z_train, y_train),
        batch_size=64,
        shuffle=True,
        num_workers=0,
        generator=loader_generator
    )
    
    # Fresh classifier for each experiment
    head = build_head(
        HEAD_EXPERIMENT,
        FEATURE_DIM,
        NUM_CLASSES
    ).to(DEVICE)
    
    criterion = nn.CrossEntropyLoss()
    
    optimizer = torch.optim.AdamW(
        head.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    print("Experiment:", HEAD_EXPERIMENT)
    print("Head seed:", SEED_HEAD)
    print("Architecture:")
    print(head)
    
    best_accuracy = -1.0
    best_weights = None
    bad_epochs = 0
    history = []
    for epoch in range(1, EPOCHS + 1):
        head.train()
        running_loss = 0.0
        for xb, yb in training_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(head(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(yb)
        head.eval()
        with torch.inference_mode():
            val_pred = head(Z_val.to(DEVICE)).argmax(dim=1).cpu().numpy()
        val_accuracy = accuracy_score(y_val.numpy(), val_pred)
        history.append({'epoch': epoch, 'train_loss': running_loss / len(Z_train),
                        'val_accuracy': val_accuracy})
        print(f'Epoch {epoch:02d}/{EPOCHS} | train loss: {history[-1]["train_loss"]:.4f} | val accuracy: {val_accuracy:.2%}')
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_weights = {key: value.detach().cpu().clone() for key, value in head.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print('Early stopping: validation accuracy did not improve')
                break
    
    head.load_state_dict(best_weights)
    head.eval()
    print(f'Best validation accuracy: {best_accuracy:.2%}')
    # pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
    
    # torch.save({'head_state_dict': best_weights, 'feature_mean': feature_mean,
    #             'feature_std': feature_std, 'class_to_idx': train_ds.class_to_idx,
    #             'backbone': MODEL_NAME}, OUTPUT_DIR / 'linear_head.pt')
    
    RUN_NAME = (
        f"F0_dinov2_vits14_p3_e4_aug{SEED_AUG}_head{SEED_HEAD}"
    )
    
    checkpoint_path = OUTPUT_DIR / f"{RUN_NAME}.pt"
    submission_path = OUTPUT_DIR / f"{RUN_NAME}.csv"
    
    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / f"{RUN_NAME}_history.csv",
        index=False
    )
    
    torch.save({
        "head_state_dict": best_weights,
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "class_to_idx": train_ds.class_to_idx,
        "backbone": MODEL_NAME,
        "preprocessing": EXPERIMENT,
        "head_experiment": HEAD_EXPERIMENT,
        "head_config": HEAD_CONFIGS[HEAD_EXPERIMENT],
        "seed_aug": SEED_AUG,
        "seed_head": SEED_HEAD,
        "best_accuracy": best_accuracy
    }, OUTPUT_DIR / f"{RUN_NAME}.pt")
    
    print("Saved experiment:", RUN_NAME)
    expected_original_accuracy=best_accuracy
    checkpoint_path=OUTPUT_DIR/f'{RUN_NAME}.pt'
    return head, history, best_accuracy, checkpoint_path

heads=[]
records=[]
for seed in HEAD_SEEDS:
    head,history,accuracy,checkpoint=train_one_head(seed)
    head.eval()
    heads.append(head)
    with torch.inference_mode():
        prediction=head(Z_val.to(DEVICE)).argmax(1).cpu().numpy()
    records.append({'seed':seed,'accuracy':float(accuracy),'macro_f1':float(f1_score(y_val.numpy(),prediction,labels=list(range(20)),average='macro',zero_division=0)),
        'correct':int((prediction==y_val.numpy()).sum()),'checkpoint':str(checkpoint)})


## 8. Fixed equal-probability ensemble
Average softmax probabilities from exactly all three heads. No fitted weights, validation-based seed selection or TTA. Test labels are never read. Export individual CSVs and the ensemble so Kaggle comparisons can isolate the change.

In [ ]:
assert HEAD_SEEDS == [42,123,2026] and len(heads)==3
with torch.inference_mode():
    val_probabilities=torch.stack([head(Z_val.to(DEVICE)).softmax(1).cpu() for head in heads])
    test_probabilities=torch.stack([head(Z_test.to(DEVICE)).softmax(1).cpu() for head in heads])
assert val_probabilities.shape==(3,200,20)
assert test_probabilities.shape==(3,len(required_ids),20)
ensemble_val=val_probabilities.mean(0)
ensemble_test=test_probabilities.mean(0)
prediction=ensemble_val.argmax(1).numpy()
ensemble_metrics={'accuracy':float(accuracy_score(y_val.numpy(),prediction)),
    'macro_f1':float(f1_score(y_val.numpy(),prediction,labels=list(range(20)),average='macro',zero_division=0)),
    'correct':int((prediction==y_val.numpy()).sum()),'count':len(y_val)}
def export_csv(name,probabilities):
    predictions=probabilities.argmax(1).numpy().astype(int)
    assert len(predictions)==len(required_ids)==len(sample)
    assert ((predictions>=0)&(predictions<20)).all()
    frame=pd.DataFrame({'image_id':required_ids,'label':predictions})
    assert frame.image_id.is_unique and frame.image_id.tolist()==sample.image_id.tolist()
    path=OUTPUT_DIR/name
    frame.to_csv(path,index=False,mode='x')
    return path
individual_csvs=[export_csv(f'F0_seed{seed}_submission.csv',test_probabilities[i]) for i,seed in enumerate(HEAD_SEEDS)]
ensemble_csv=export_csv('F0_three_seed_ensemble_submission.csv',ensemble_test)
summary={'head_seeds':HEAD_SEEDS,'augmentation_seed':SEED_AUG,'weights':[1/3,1/3,1/3],
    'individual_validation':records,'ensemble_validation':ensemble_metrics,
    'combination':'arithmetic mean of all three probability vectors','TTA':False,
    'feature_signature':FEATURES_SIGNATURE,'backbone':MODEL_NAME,'hub_ref':HUB_REF,
    'weights_sha256':WEIGHTS_SHA256,'prediction_count':len(required_ids),'submitted':False,
    'individual_csvs':[str(p) for p in individual_csvs],'ensemble_csv':str(ensemble_csv),
    'validation_gate_for_export':False,'historical_Kaggle_reference':0.95510}
(OUTPUT_DIR/'ensemble_summary.json').write_text(json.dumps(summary,indent=2))
pd.DataFrame(records).to_csv(OUTPUT_DIR/'individual_validation.csv',index=False)
np.savez_compressed(OUTPUT_DIR/'validation_probabilities.npz',seeds=np.asarray(HEAD_SEEDS),individual=val_probabilities.numpy(),ensemble=ensemble_val.numpy(),targets=y_val.numpy())
display(pd.DataFrame(records))
print('Ensemble validation:',ensemble_metrics)
print('Kaggle candidate (not submitted):',ensemble_csv)
print('Same-run seed2026 comparison:',individual_csvs[2])
assert not backbone.training and not any(p.requires_grad for p in backbone.parameters())
assert weight_digest(backbone)==WEIGHTS_SHA256


## 9. What to submit
Use F0_three_seed_ensemble_submission.csv as the new candidate. Keep the seed2026 file from this run as its matched control; a fresh run may differ from your previous0.95510 model. There is no claim of Kaggle improvement until measured. Preserve the entire run directory and feature cache in Drive. The200-image validation set remains a sanity check; no leaderboard-driven weight search is implemented.